In [19]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

import warnings
warnings.filterwarnings("ignore")

In [20]:
## Loading all the saved models

model = load_model('churn_model.h5')

## Loading scaler model
with open('scaler.pkl','rb') as f:
    scaler = pickle.load(f)

## Loading gender encoder model
with open('encoder_gender.pkl','rb') as f:
    gender_encoder = pickle.load(f)

## Loading Geography encoder model
with open('encoder_geography.pkl','rb') as f:
    geography_encoder = pickle.load(f)

In [21]:
## Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


## Create a function to preprocess the input data
def preprocess_input(data, gender_encoder=gender_encoder, geography_encoder=geography_encoder, scaler=scaler):

    '''Preprocess the input data for prediction.
    Args:
        data (dict): A dictionary containing the input features.
        gender_encoder (LabelEncoder): Fitted LabelEncoder for Gender.
        geography_encoder (OneHotEncoder): Fitted OneHotEncoder for Geography.
        scaler (StandardScaler): Fitted StandardScaler for scaling the data.
    Returns:
        np.ndarray: Preprocessed and scaled input data ready for prediction.
    '''

    ## Covert the data into a DataFrame
    input_df = pd.DataFrame([data])

    ## Encode the Gender column
    input_df['Gender'] = gender_encoder.transform(input_df['Gender'])

    ## Encode the Geography data
    encoded_geo = geography_encoder.transform([[data['Geography']]])
    encoded_geo_df = pd.DataFrame(encoded_geo,columns=geography_encoder.get_feature_names_out(['Geography']))

    ## Drop the original Geography column and concatenate the encoded columns
    final_input_df = pd.concat([input_df.drop('Geography',axis=1),encoded_geo_df],axis=1)

    print("Final input DataFrame after encoding:")
    display(final_input_df)

    ## scale the data
    scaled_input_df = scaler.transform(final_input_df)

    return scaled_input_df

In [22]:
## Preprocess the input data with the function
preprocessed_data = preprocess_input(input_data)

Final input DataFrame after encoding:


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [25]:
probability = model.predict(preprocessed_data)
probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


array([[0.01889914]], dtype=float32)

In [26]:
## Predicting the output
def predict_churn(model, preprocessed_data):
    '''Predict churn using the preprocessed data.
    Args:
        model (tf.keras.Model): Trained Keras model for prediction.
        preprocessed_data (np.ndarray): Preprocessed input data.
    Returns:
        str: Prediction result indicating whether the customer will churn or not.
    '''
    prediction = model.predict(preprocessed_data)
    print(f"Raw model prediction output: {prediction}")

    ## Convert probability to class label
    predicted_class = (prediction > 0.5).astype(int)

    return "Customer will Churn" if predicted_class[0][0] > 0.5 else "Customer will Not Churn"


predict_churn(model, preprocessed_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Raw model prediction output: [[0.01889914]]


'Customer will Not Churn'